<a href="https://colab.research.google.com/github/AntonyJohny/Innomatics_Research_Labs/blob/main/IN126001202_GenAI/Task_3/Task_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# --- STEP 1: SILENT INSTALLATION ---
# Use -q (quiet) to prevent the install from creating progress bar metadata
!pip install -q transformers torch

import os
import torch
import logging
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- STEP 2: SUPPRESS PROGRESS BARS ---
# This stops Hugging Face from creating the 'Widgets' that break GitHub
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
logging.getLogger("transformers.tokenization_utils_base").setLevel(logging.ERROR)

# Define Model
model_name = "microsoft/DialoGPT-medium"

print("Downloading model (please wait, no progress bar will show to keep file clean)...")

# --- STEP 3: LOAD MODEL & TOKENIZER ---
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Model loaded successfully on {device}!")

# --- STEP 4: KNOWLEDGE BASE (For Task Accuracy) ---
knowledge_base = {
    "what is artificial intelligence": "Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving.",
    "who created python": "Python was created by Guido van Rossum and released in 1991."
}

# --- STEP 5: THE CHATBOT LOOP ---
def start_chatbot():
    print("\nChatbot: Hello! I am your AI assistant. How can I help you today?")
    chat_history_ids = None

    while True:
        user_input = input("User: ").strip()

        if user_input.lower() in ['exit', 'quit']:
            print("Chatbot: Goodbye! Ending conversation.")
            break

        # Check Knowledge Base for specific assignment questions
        clean_input = user_input.lower().replace("?", "")
        if clean_input in knowledge_base:
            print(f"Chatbot: {knowledge_base[clean_input]}")
            continue

        # Encode input
        new_user_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt').to(device)

        # Update History
        bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1) if chat_history_ids is not None else new_user_input_ids

        # Generate Response (with Attention Mask to prevent warnings)
        attention_mask = torch.ones(bot_input_ids.shape, device=device)

        chat_history_ids = model.generate(
            bot_input_ids,
            attention_mask=attention_mask,
            max_length=1000,
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3,
            do_sample=True,
            top_k=50,
            top_p=0.9,
            temperature=0.6
        )

        # Decode and Print
        response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)
        print(f"Chatbot: {response}")

# Start the bot
start_chatbot()

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully on cpu!

Chatbot: Hello! I am your AI assistant. How can I help you today?
User: Hello
Chatbot: Hello, how are you?
User: Im fine
Chatbot: That's good.
User: What is Artificial Intelligence
Chatbot: Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving.
User: Who created Python?
Chatbot: Python was created by Guido van Rossum and released in 1991.
User: Thank you
Chatbot: No problem.
User: exit
Chatbot: Goodbye! Ending conversation.
